# 🫀 퀘스트 46 · Q3 — **무라벨 EM 사전확률 보정**(층② 눈금의 첫 처방)

| | **MedKOS / `notebooks/quest46_q3_prior_em.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 맥락정의 클래스(S)의 평가·보정 프로토콜 |
| 층 | **② 점수 눈금** (층① 표현은 Q8, 층③ 채점은 R11 로 확립) |
| 방법 | Saerens · Latinne · Decaestecker (2002) EM prior adjustment |
| 부모 런 | `quest46_q7aa_burden_target` · `quest46_q7s3_recompute` |
| 관문 | **C0~C7 사전등록 완료** — 이 노트북은 설계가 아니라 **구현**이다 |

## 이 런이 묻는 것

> 기록마다 S 유병률이 다르다. 그래서 **점수의 눈금이 기록마다 다르고**, 환자 간
> 점수 비교가 깨진다(실험22-A: 전역 하락의 **95.4%** 가 눈금 탓). **무라벨 EM 으로
> 각 기록의 사전확률을 추정해 로짓을 옮기면 그 눈금이 고쳐지는가.**

## ★★★ 빠뜨리면 이 런이 무효인 셋

1. **C2 를 C3 보다 먼저 읽는다.** 오라클 사전확률 팔(진짜 유병률로 보정)이 자기 MDE 를
   못 넘으면 **방법 A 는 그 자리에서 답이 난다** — 눈금 붕괴가 사전확률 이동으로
   설명되지 않는다는 뜻이고, 그러면 **C3 의 회복률(비)을 읽지 않는다**(R40 ① · R41 ②).
2. **기저 모델 보정은 DEV 에서만 적합한다.** EM 은 **보정된 사후확률**을 전제한다.
   기저가 안 맞으면 `π̂` 가 쓰레기이고, C3 실패가 **방법 탓인지 기저 탓인지** 못 가른다.
3. **매크로는 판정 지표가 아니라 항등 대조다.** 사전확률 보정은 레코드 안에서
   **상수 로짓 시프트**라 레코드 내 순위를 보존한다 → 레코드별 AUROC·PR-AUC 가
   **정의상 불변**이다. 움직이면 **버그이거나 비단조**이고, 그때는 아래를 안 읽는다.

## 주 지표 — 왜 전역(pooled)인가

R11 은 전역 지표를 **보조**로 격하했다. Q3 은 그 예외이고, 이유가 있다:
**Q3 의 표적 자체가 「환자 간 점수 비교 가능성」이고, 보정이 움직일 수 있는 것도
그것뿐**이다(레코드 내 순위는 보존되므로). 따라서 **Q3 한정으로 주 지표는 전역 PR-AUC**,
매크로는 **항등 대조**다. 대신 R11 의 조건을 지킨다 — 지배 지분·제외 레코드·`GMIN_S` 를
**항상 병기**하고 전역 수치를 단독 인용하지 않는다.

## 회복률의 분모 (사전에 못 박았다)

원문 「전역 PR-AUC 회복 ≥ 50%」가 분모를 안 밝혔으므로 **분모 = 오라클 팔의 이득**으로
고정한다. 그리고 **절대 ΔPR-AUC 를 레코드 군집 부트스트랩 CI 와 함께 항상 병기**한다 —
비만 보면 분모가 0 근처일 때 CI 가 폭발한다(**R40 ② · R41 ②** 가 정확히 이걸로 데였다).

```
회복률 = (PRAUC_EM − PRAUC_raw) / (PRAUC_oracle − PRAUC_raw)
```

## 관문

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **C0** | 항등 대조 — `π_r := π_tr` 이면 정확히 항등 | `max\|Δscore\| = 0`. **구성으로 증명** |
| **C1** | 순위 보존 항등 — 레코드별 AUROC·PR-AUC 불변 | `max\|Δ\| < 1e-12`. 깨지면 **중단** |
| **C2 ★★** | **오라클 사전확률 팔** — 전제의 직접 검정 | **먼저 읽는다.** 이득 > 자기 MDE **그리고** CI 가 0 을 뗄 것 |
| **C3 ★★★** | **EM 팔** — 절대 ΔPR-AUC + 오라클 대비 회복률 | Δ 의 CI 가 0 을 뗄 것 **그리고** 회복률 ≥ 50%. 하나만이면 **미결** |
| **C4** | 영점 — 학습 라벨 치환 뒤 **같은 EM** | **측정된** 영점 안(가정 금지) |
| **C5** | `π̂` 진단 + 기저 보정도(ECE) | 관문 아님. **C2 ✅ · C3 ❌** 일 때 「방법 vs 추정」을 가른다 |
| **C6** | 매크로 — 단조 팔은 C1 로 항등 | 팔별 단조성 사전등록 |
| **C7** | 결론 검산표 코드 고정 | R38 ⑦ · R39 ⑤ |

### 판정표 (갈래의 운명)

- **C2 ❌** → 눈금 붕괴는 사전확률 이동이 아니다. **방법 A 종결 → Q4**
- **C2 ✅ · C3 ✅** → 처방 확보. Q4 는 **A 대비 증분**으로 판정
- **C2 ✅ · C3 ❌ · C5 가 `π̂` 를 지목** → 병목은 **추정** → 추정기 교체(BBSE·MLLS)
- **C2 ✅ · C3 ❌ · C5 가 `π̂` 무죄** → **모형 오설정**(label shift 가정 실패) → 한계 명시 후 Q4

⚠️ **새 데이터 0** — `svdb_data5.npz` 만 쓴다. P 위치 자산(Q7-P0)도 BUT PDB 도 안 쓴다:
**λ 는 이 런의 질문이 아니다.**


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def spearman(a, b):
    ra = np.asarray(a, float).argsort().argsort().astype(float)
    rb = np.asarray(b, float).argsort().argsort().astype(float)
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])

def need_super(n, half, eff, p80=False):
    """★ 우월(superiority) 프레임. 등가와 **다른 수**이므로 프레임을 밝혀서 쓴다(R37 ①).
    ⚠️ 효과가 0 근처면 이 수는 **해석 불가**다 — 호출부에서 그렇게 찍는다(R41 ②)."""
    if not np.isfinite(half) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def ece_of(p, y, nbin=15):
    """기대 보정 오차(equal-width). C5 에서 **기저가 보정돼 있나**를 본다."""
    p = np.asarray(p, float); y = np.asarray(y, float)
    edges = np.linspace(0.0, 1.0, nbin + 1)
    e = 0.0
    for i in range(nbin):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < nbin - 1 else p <= edges[i + 1])
        if not m.any():
            continue
        e += (m.mean()) * abs(p[m].mean() - y[m].mean())
    return float(e)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

# ★ 스모크런 훅 — **비용 손잡이만** 만진다. 관문 문턱·설계 상수는 절대 안 건드린다.
SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25              # ★ GMIN_S — 채점 가능 레코드 조건(R11-b)

# ── 비용 손잡이(스모크에서만 축소)
NB_BOOT = 200 if SMOKE else 1000   # 레코드 군집 부트스트랩 반복
N_PERM  = 3   if SMOKE else 50     # ★ 영점 reps — R39 ① 이 확정한 하한은 50

# ── ★★★ 사전등록 ①: 판정을 **읽는 순서**. C2 가 C3 보다 먼저다
READ_ORDER = ("C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7")
GATE_DEP = {
    "C1": ["C0"],
    "C2": ["C0", "C1"],
    "C3": ["C0", "C1", "C2"],   # ★ C2 가 ❌ 면 C3 의 **비(회복률)를 읽지 않는다**
}

# ── ★★★ 사전등록 ②: 주 지표와 매크로의 역할
PRIMARY = "pooled_prauc"           # 전역 PR-AUC. 보정이 움직일 수 있는 건 이것뿐이다
MACRO_ROLE = "identity_control"    # ★ 매크로는 판정 지표가 아니라 **항등 대조**다

# ── ★★★ 사전등록 ③: 팔별 **단조성**. 단조 팔은 C1 로 항등, 비단조 팔만 매크로 판정 대상
ARM_MONOTONE = {
    "raw":      True,   # 보정 없음(기준)
    "identity": True,   # π_r := π_tr — **정확히 항등**이어야 한다(C0)
    "oracle":   True,   # π_r := 진짜 유병률 — 상수 로짓 시프트
    "em":       True,   # π_r := EM 추정 — 상수 로짓 시프트
}
# ⚠️ 이 런에는 **비단조 팔이 없다** → C6 의 매크로는 전부 C1 의 항등으로 처리된다.
#    비단조 팔(클리핑·레코드별 임계값)을 넣는 후속 런에서만 매크로가 진짜 판정 대상이 된다.

# ── 회복률 분모를 **여기서 못 박는다**(R39 ①)
RECOVERY_DEN = "oracle_gain"       # 회복률 = (EM − raw) / (oracle − raw)
RECOVERY_THR = 0.50                # 사전등록 합격선
TOL_IDENT = 1e-12                  # C1 항등 허용오차
# ★★ 부담순으로 세운 뒤 이 20칸 패턴을 돌려 배분한다(TRAIN 8 · DEV 5 · TEST 7 = 40/25/35).
#    ⚠️ 패턴은 반드시 **뒤섞여** 있어야 한다 — `TRAIN*8 + DEV*5 + TEST*7` 처럼 **덩어리**로
#    두면 부담순 배분이 「TRAIN=저부담 / DEV=중간 / TEST=고부담」 **블록 분할**이 된다.
#    1판이 정확히 그렇게 죽었다(TEST 유병률이 0.25~0.32 로만 차서 눈금 문제가 사라졌다).
#    Q3 은 **유병률이 벌어져야** 질문이 성립하므로 각 분할이 범위를 고루 덮어야 한다.
SPLIT_PATTERN = ("TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN", "TEST", "TRAIN", "DEV", "TEST",
                 "TRAIN", "TEST", "DEV", "TRAIN")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    macro_auroc=0.8842, macro_lo=0.8449, macro_hi=0.9175, macro_n=72,  # Q7-B′
    calib_share=0.954,          # 눈금이 전역 하락에서 차지하는 몫(실험22-A)
    ranknorm_before=0.5138, ranknorm_after=0.0851,   # 순위정규화의 대가(07-17)
    dominant=0.149, gmin10=66,  # Q7-A
    base_auprc=0.416)           # 리듬-only 교차환자 AUPRC(Q7-S″ / Q7-AA)

RULE_CHECK = {
    "R11 / R11-b":        "전역이 주 지표인 **예외 런** — 지배 지분·제외 레코드·GMIN_S 를 병기하고 단독 인용 금지",
    "R16 fallback 없음":  "자산 없으면 **중단**",
    "R22 누출 없음":      "★★ 기저 보정·EM 하이퍼를 **DEV 에서만** 고정 — 보고할 지표로 고르면 누출",
    "R26 / R38 ②":        "영점은 **측정**한다 — 0 을 가정하지 않는다",
    "R29 ② 분기 금지":    "★★ C0·C1 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":          "관문마다 MDE 를 내고 점추정과 비교. **미결 ≠ 등가**",
    "R34 ③ 대조 보장":    "★★ C0 은 **구성으로 항등** — 가정이 아니라 항등식이다",
    "R35 ④ 런타임 검사":  "항등이 깨지면 판정하지 말고 **멈춘다**",
    "R36 ② 선택은 DEV":   "보정기(Platt/isotonic) 선택도 **DEV 반쪽**에서",
    "R37 ① 프레임":       "필요표본은 **우월 프레임**임을 밝히고 50%/80% 를 함께",
    "R39 ① 분모 고정":    "★ 회복률 분모 = **오라클 팔의 이득** — 사후에 고르지 않는다",
    "R40 ① 직접 검정":    "★★ C2 가 **label shift 가정의 직접 검정**이다 — 먼저 읽는다",
    "R41 ② 0 근처":       "★ 효과가 0 근처면 필요표본은 **해석 불가**라고 같이 찍는다",
}

CONFIG = dict(
    exp="quest46_q3_prior_em", quest="ailab-2026-0046", step="prior-em-calibration",
    parent_exp=["quest46_q7aa_burden_target", "quest46_q7s3_recompute"],
    method="Saerens · Latinne · Decaestecker (2002) — EM prior adjustment",
    purpose=("**층② 눈금의 첫 처방.** 기록마다 S 유병률이 달라 점수 눈금이 어긋나고, "
             "실험22-A 는 그 눈금이 전역 하락의 **95.4%** 를 설명한다고 진단했다. "
             "순위정규화는 **처방으로 못 쓴다**(PR-AUC 0.5138 → 0.0851 로 절대 성능이 폭락). "
             "그래서 **무라벨 EM 사전확률 보정**(방법 A)을 시험한다. "
             "★★★ **C2(오라클 사전확률 팔)를 C3 보다 먼저 읽는다** — 진짜 유병률로 "
             "보정해도 전역 PR-AUC 가 안 오르면 눈금 붕괴는 사전확률 이동이 아니고, "
             "**방법 A 는 그 자리에서 답이 난다**(그러면 C3 의 회복률을 읽지 않는다). "
             "★★ 기저 모델 보정은 **DEV 에서만** 적합한다 — EM 은 보정된 사후확률을 "
             "전제하므로, 안 하면 C3 실패가 **방법 탓인지 기저 탓인지** 못 가른다. "
             "★★ **매크로는 판정 지표가 아니라 항등 대조**다 — 보정이 레코드 내 순위를 "
             "보존하므로 레코드별 지표는 **정의상 불변**이고, 움직이면 버그다."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · P 위치 자산도 BUT PDB 도 쓰지 않는다)",
    primary=PRIMARY, macro_role=MACRO_ROLE, arm_monotone=ARM_MONOTONE,
    read_order=READ_ORDER, gate_dep=GATE_DEP,
    recovery_den=RECOVERY_DEN, recovery_thr=RECOVERY_THR,
    n_boot=NB_BOOT, n_perm=N_PERM, smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "C0": "**항등 대조** — `π_r := π_tr` 이면 보정이 **정확히 항등**. 로짓 시프트를 "
              "`Δ = 0` 으로 만들어 **구성으로** 통과시킨다. `max|Δscore|` 가 0 이 아니면 **중단**",
        "C1": "**순위 보존 항등** — 상수 로짓 시프트는 레코드 내 순위를 보존하므로 "
              f"레코드별 AUROC·PR-AUC 가 불변이다. `max|Δ| < {TOL_IDENT}`. "
              "움직이면 **버그이거나 비단조** → 원인 규명 전엔 아래를 안 읽는다",
        "C2": "★★ **오라클 사전확률 팔 — 여기를 먼저 읽는다.** 진짜 유병률 `π*_r` 로 "
              "보정하면 전역 PR-AUC 가 오르나. 이득이 **자기 MDE 를 못 넘으면** 방법 A 는 "
              "그 자리에서 답이 나고 **C3 의 비를 읽지 않는다**(R40 ① · R41 ②)",
        "C3": "★★★ **주 관문 — EM 팔.** 절대 ΔPR-AUC(레코드 군집 부트스트랩 CI) **그리고** "
              f"오라클 대비 회복률 ≥ {RECOVERY_THR:.0%}. **하나만 만족하면 미결**로 쓴다",
        "C4": "**영점** — 학습 라벨을 치환해 정보 없는 기저 점수를 만든 뒤 **같은 파이프라인**"
              f"(DEV 보정 → 레코드별 EM)을 밟는다(reps={N_PERM}). 0 을 **가정하지 않는다**. "
              "⚠️ 오라클 팔은 **진짜 유병률**을 주입하므로 영점에서도 이득이 날 수 있다 — "
              "그게 바로 이 관문이 필요한 이유다",
        "C5": "**관문 아님.** `|π̂_r − π*_r|` 분포·상관 + 기저 **ECE·신뢰도 곡선**. "
              "**C2 ✅ 인데 C3 ❌** 이면 병목이 「방법」인지 「추정」인지를 여기서 가른다",
        "C6": "**매크로** — 이 런의 팔은 **전부 단조**라 C1 의 항등으로 처리된다. "
              "판정 지표가 아니다. 비단조 팔을 넣는 후속 런에서만 진짜 판정 대상이 된다",
        "C7": "**결론 검산표** — 판정마다 (a) 근거 숫자 (b) 미검정 가정 (c) 틀리면 어떻게 바뀌나"},
    caveat=("★★ **label shift 가정을 명시한다.** EM prior adjustment 는 `p(x|y)` 불변을 "
            "가정하는데 SVDB 는 환자마다 형태가 달라 **covariate shift 도 있다** — 즉 가정이 "
            "어긋날 수 있고, **C2 가 바로 그 가정의 직접 검정**이다(R40 ①). "
            "★ **오라클 팔은 방법이 아니라 상한**이다 — TEST 라벨의 유병률을 쓰므로 배포 "
            "가능한 절차가 아니다. 그래서 **상한으로만** 인용한다. "
            "★ 전역 지표는 **유병률에 민감**하므로 지배 지분·제외 레코드를 항상 병기한다(R11 · R32). "
            "★ **새 데이터 0** — svdb_data5.npz 만. λ 는 이 런의 질문이 아니다")
)
np.random.seed(SEED0)
run = MedKOSRun("quest46_q3_prior_em", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q3 — 무라벨 EM 사전확률 보정(층② 눈금)**")
run.log(f"  주 지표 = **전역 PR-AUC** · 매크로 = **항등 대조**(판정 지표가 아니다)")
run.log(f"  회복률 분모 = **오라클 팔의 이득** (사전 고정 · 합격선 {RECOVERY_THR:.0%})")
run.log("  ★★★ **C2 를 C3 보다 먼저 읽는다** — C2 ❌ 면 C3 의 비를 읽지 않는다")
run.log("  ★★ 기저 보정은 **DEV 에서만** 적합 — EM 은 보정된 사후확률을 전제한다")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM}). "
            "관문 문턱은 그대로다")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【Q3-0】 자산 · 코호트 · TRAIN/DEV/TEST · 기저 모델 · ★★ DEV 전용 보정
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【Q3-0】 코호트 · 레코드 분할 · 기저 모델 · **DEV 전용 보정**")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음 — svdb_labels.py build(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

# ── 리듬 특징 (Q7 과 동일 · 새 데이터 0)
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
            np.log1p(pre), np.log1p(post)]
RHY = np.nan_to_num(RHY, nan=0.0, posinf=0.0, neginf=0.0)

# ── 채점 가능 코호트 (GMIN_S · R11-b) — 제외 레코드를 **세어서** 남긴다
IDXS_ALL = {r: np.where(RID == r)[0] for r in RS}
REC_OK = [r for r in RS
          if TT[IDXS_ALL[r]].sum() >= MIN_S and (~TT[IDXS_ALL[r]]).sum() >= MIN_N]
EXCL = [int(r) for r in RS if r not in REC_OK]
BURD_ALL = {int(r): float(TT[IDXS_ALL[r]].mean()) for r in RS}
run.log(f"  레코드 {len(RS)} · 채점 가능 {len(REC_OK)} · 제외 {len(EXCL)} "
        f"(GMIN_S = S≥{MIN_S} & N≥{MIN_N})")
run.log(f"  제외 레코드: {EXCL if EXCL else '없음'}")

# ── ★ 부담순 결정 배분(무작위 없음) — 각 분할이 **유병률 범위를 고루** 덮게
order = sorted(REC_OK, key=lambda r: (BURD_ALL[int(r)], int(r)))
SPLIT = {int(r): SPLIT_PATTERN[i % len(SPLIT_PATTERN)] for i, r in enumerate(order)}
TR_R = [r for r in REC_OK if SPLIT[int(r)] == "TRAIN"]
DV_R = [r for r in REC_OK if SPLIT[int(r)] == "DEV"]
TE_R = [r for r in REC_OK if SPLIT[int(r)] == "TEST"]
if min(len(TR_R), len(DV_R), len(TE_R)) < 3:
    raise AssetError(f"분할이 너무 얕다 — TRAIN {len(TR_R)} · DEV {len(DV_R)} · TEST {len(TE_R)}")
sel = lambda rs: np.concatenate([IDXS_ALL[r] for r in rs])
TR_I, DV_I, TE_I = sel(TR_R), sel(DV_R), sel(TE_R)
SPREAD = {}
for nm, rr, ii in (("TRAIN", TR_R, TR_I), ("DEV", DV_R, DV_I), ("TEST", TE_R, TE_I)):
    bs = [BURD_ALL[int(r)] for r in rr]
    SPREAD[nm] = (float(min(bs)), float(max(bs)))
    run.log(f"  {nm:<6}레코드 {len(rr):>3} · 비트 {len(ii):>7,} · S 유병률 "
            f"{TT[ii].mean():.4f} (레코드별 {min(bs):.4f}~{max(bs):.4f})")

# ── ★★ 분할이 **유병률 범위를 고루** 덮는지 검사한다. Q3 의 질문 자체가 유병률 차이에
#    걸려 있으므로, TEST 가 좁은 띠만 담으면 **눈금 문제가 사라져** 관문이 무의미해진다.
ALL_LO, ALL_HI = min(BURD_ALL[int(r)] for r in REC_OK), max(BURD_ALL[int(r)] for r in REC_OK)
cover = (SPREAD["TEST"][1] - SPREAD["TEST"][0]) / max(ALL_HI - ALL_LO, 1e-9)
run.log(f"  ★ TEST 유병률 커버리지 **{cover:.2f}** (코호트 범위 {ALL_LO:.4f}~{ALL_HI:.4f} 대비)")
if cover < 0.5:
    raise AssetError(
        f"분할이 유병률을 **블록으로 갈랐다**(TEST 커버리지 {cover:.2f} < 0.5). "
        "부담순 배분에 덩어리 패턴을 쓰면 이렇게 된다 — SPLIT_PATTERN 이 섞여 있어야 한다. "
        "이 상태로는 Q3 의 질문(기록마다 다른 유병률)이 성립하지 않는다")

# ── ★ R11 — 전역이 주 지표인 예외 런이므로 **지배 지분**을 반드시 병기한다
s_cnt = np.array([int(TT[IDXS_ALL[r]].sum()) for r in TE_R], float)
DOMINANT = float(s_cnt.max() / s_cnt.sum())
run.log(f"  ★ TEST 지배 지분 **{DOMINANT:.3f}** (한 레코드가 S 의 이만큼) · "
        f"참고 전수 코호트 {REF['dominant']:.3f}")
run.log("    ▸ **전역 수치를 단독 인용하지 않는다** — 지배 지분·제외 레코드와 함께만(R11)")

# ── 기저 모델: TRAIN 에서만 적합
def fit_base(tr_idx, ytr):
    mu, sd = RHY[tr_idx].mean(0), RHY[tr_idx].std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((RHY[tr_idx] - mu) / sd, ytr)
    return lambda idx: lr.decision_function((RHY[idx] - mu) / sd)

score_of = fit_base(TR_I, TT[TR_I].astype(int))
SC_dev, SC_te = score_of(DV_I), score_of(TE_I)

# ── ★★★ 보정은 **DEV 에서만** 적합한다. TEST 는 절대 안 본다
#    선택(Platt vs isotonic)도 **DEV 반쪽**에서 — 보고할 지표로 고르면 누출이다(R22 · R36 ②)
half = len(DV_R) // 2
DV_FIT_R, DV_SEL_R = DV_R[:half], DV_R[half:]
if not DV_FIT_R or not DV_SEL_R:
    raise AssetError("DEV 반쪽이 안 나뉜다 — 보정기 선택을 지표로 하게 되므로 중단")
in_ = lambda rs: np.isin(RID[DV_I], np.asarray(rs))
m_fit, m_sel = in_(DV_FIT_R), in_(DV_SEL_R)

def make_platt(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6)
    lr.fit(np.asarray(s).reshape(-1, 1), np.asarray(y).astype(int))
    return lambda v: lr.predict_proba(np.asarray(v).reshape(-1, 1))[:, 1]

def make_iso(s, y):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6)
    ir.fit(np.asarray(s), np.asarray(y).astype(float))
    return lambda v: np.clip(ir.predict(np.asarray(v)), 1e-6, 1 - 1e-6)

CAND = {}
for nm, mk in (("platt", make_platt), ("isotonic", make_iso)):
    f_ = mk(SC_dev[m_fit], TT[DV_I][m_fit])
    CAND[nm] = ece_of(f_(SC_dev[m_sel]), TT[DV_I][m_sel].astype(float))
    run.log(f"  보정기 후보 {nm:<9} DEV-sel(홀드아웃) ECE {CAND[nm]:.5f}")
CAL_NAME = min(CAND, key=lambda k: CAND[k])
run.log(f"  ★ 선택 = **{CAL_NAME}** (DEV 반쪽 ECE 기준 — TEST 를 안 봤다)")
calib = {"platt": make_platt, "isotonic": make_iso}[CAL_NAME](SC_dev, TT[DV_I])

P_dev, P_te = calib(SC_dev), calib(SC_te)
EPS = 1e-6
P_dev = np.clip(P_dev, EPS, 1 - EPS); P_te = np.clip(P_te, EPS, 1 - EPS)

# ── ★★ π_tr — 보정기가 DEV 에서 적합됐으므로 사후확률은 **DEV 유병률**로 보정돼 있다.
#    따라서 EM 의 기준 사전확률은 DEV 유병률이다(학습 유병률이 아니다).
PI_TR = float(TT[DV_I].mean())
run.log(f"  π_tr = **DEV 유병률 {PI_TR:.5f}** (보정기를 DEV 에서 적합했으므로 기준이 여기다)")
run.log(f"    검산 — DEV 보정 사후확률 평균 {P_dev.mean():.5f} (π_tr 과 같아야 한다)")
# ★★ DEV ECE 는 **홀드아웃 값**(DEV 반쪽)을 쓴다. 최종 보정기를 DEV 전체에 다시 적합했으므로
#    같은 DEV 에서 잰 ECE 는 **구성상 0 에 가깝다**(특히 isotonic) — 그 수를 보정도의 증거로
#    쓰면 요약이 사실과 어긋난다(R38 ⑦).
ECE_DEV = CAND[CAL_NAME]
ECE_DEV_IN = ece_of(P_dev, TT[DV_I].astype(float))
ECE_TE = ece_of(P_te, TT[TE_I].astype(float))
run.log(f"  기저 보정도 — ECE **DEV(홀드아웃) {ECE_DEV:.5f}** · TEST {ECE_TE:.5f} "
        f"(C5 에서 다시 읽는다)")
run.log(f"    ▸ 참고 DEV 자기적합 ECE {ECE_DEV_IN:.5f} — 자기적합이라 **낙관적이다**"
        "(isotonic 은 구성상 0 에 가깝다). **증거로 쓰지 않는다**")

CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=len(REC_OK), excluded=EXCL,
                        gmin_s=MIN_S, gmin_n=MIN_N, dominant=DOMINANT,
                        n_train=len(TR_R), n_dev=len(DV_R), n_test=len(TE_R),
                        pi_tr=PI_TR, calibrator=CAL_NAME, cand_ece=CAND,
                        ece_dev=ECE_DEV, ece_dev_insample=ECE_DEV_IN, ece_test=ECE_TE,
                        spread=SPREAD, test_coverage=float(cover))
run.save_json("config", CONFIG)
run.log("  Q3-0 ✅ 코호트·분할·기저·보정 준비 완료")


In [ ]:
# CELL 3 — 【Q3-A】 ★ C0 항등 대조 · C1 순위 보존 항등 (구성으로 증명 · 깨지면 중단)
run.log("\n" + "=" * 100)
run.log("【Q3-A】 C0 — 항등 대조 · C1 — 순위 보존 항등")
run.log("=" * 100)

# ── ★★ 보정을 **로짓 공간의 상수 시프트**로 구현한다.
#    p' = (w·p) / (w·p + v·(1−p)),  w = π/π_tr,  v = (1−π)/(1−π_tr)
#    ⇔ logit(p') = logit(p) + [logit(π) − logit(π_tr)]
#    확률 공간에서 계산하면 포화(p→0,1)로 동순위가 생겨 순위가 흔들린다.
#    로짓 공간의 **덧셈**이면 순위 보존이 **부동소수점에서도 정확**하다 → C0·C1 이 구성으로 선다.
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

def shift_logit(pi, pi_tr):
    """레코드 하나에 붙는 **상수** 로짓 시프트. π = π_tr 이면 정확히 0.0 을 낸다."""
    if pi == pi_tr:
        return 0.0                      # ★ C0 을 **구성으로** 보장한다(가정이 아니다)
    return float(logit(pi) - logit(pi_tr))

L_raw = logit(P_te)
RID_te = RID[TE_I]; Y_te = TT[TE_I].astype(int)
TE_POS = {int(r): np.where(RID_te == r)[0] for r in TE_R}
PI_STAR = {int(r): float(Y_te[TE_POS[int(r)]].mean()) for r in TE_R}   # 오라클용 진짜 유병률

def apply_arm(pi_by_rec):
    """레코드별 π 를 받아 **레코드 안 상수 시프트**를 더한다(비단조 연산 없음)."""
    out = L_raw.copy()
    for r in TE_R:
        pos = TE_POS[int(r)]
        out[pos] = out[pos] + shift_logit(pi_by_rec[int(r)], PI_TR)
    return out

L_ident = apply_arm({int(r): PI_TR for r in TE_R})

# ── C0 — 항등 대조. **정확히 0** 이어야 한다
d0 = float(np.max(np.abs(L_ident - L_raw)))
run.log(f"  C0 — `π_r := π_tr` 팔의 max|Δscore| = **{d0:.1e}**")
if d0 != 0.0:
    raise AssetError(f"C0 실패 — 항등이 정확히 0 이 아니다({d0:.3e}). "
                     "대조군은 가정이 아니라 **항등식**이어야 한다(R34 ③ · R35 ④)")
g_("C0", "✅ 지지", "구성으로 항등이다 — 시프트가 π=π_tr 에서 **정확히 0.0** 을 낸다")

# ── C1 — 순위 보존 항등. 레코드별 AUROC·PR-AUC 가 불변이어야 한다
def per_record(scores):
    ap, au = {}, {}
    for r in TE_R:
        pos = TE_POS[int(r)]; yy = Y_te[pos]
        if yy.sum() == 0 or yy.sum() == len(yy):
            continue
        ap[int(r)] = float(average_precision_score(yy, scores[pos]))
        au[int(r)] = float(roc_auc_score(yy, scores[pos]))
    return ap, au

AP_raw_rec, AU_raw_rec = per_record(L_raw)
L_oracle = apply_arm(PI_STAR)
worst = 0.0; worst_where = ""
for nm, LL in (("identity", L_ident), ("oracle", L_oracle)):
    ap_, au_ = per_record(LL)
    for tag, a, b in (("PR-AUC", ap_, AP_raw_rec), ("AUROC", au_, AU_raw_rec)):
        dd = max(abs(a[k] - b[k]) for k in b) if b else 0.0
        if dd > worst:
            worst, worst_where = dd, f"{nm}/{tag}"
run.log(f"  C1 — 레코드별 지표의 max|Δ| = **{worst:.1e}** (최악 {worst_where or '없음'}) "
        f"· 허용 {TOL_IDENT:.0e}")
if worst >= TOL_IDENT:
    raise AssetError(f"C1 실패(max|Δ| {worst:.3e}) — 보정이 레코드 내 순위를 바꿨다. "
                     "**버그이거나 비단조**다. 원인 규명 전에는 C2·C3 을 읽지 않는다(R29 ②)")
g_("C1", "✅ 지지",
   f"상수 로짓 시프트라 레코드 내 순위가 보존된다 — 매크로는 **항등 대조**이지 판정 지표가 아니다")
run.log(f"    ▸ 그래서 사전등록의 「매크로가 안 떨어질 것」은 **정보량 0 인 항등 관문**이다.")
run.log(f"    ▸ 이 런의 팔은 전부 단조({', '.join(k for k, v in ARM_MONOTONE.items() if v)}) "
        f"→ C6 은 판정이 아니라 검산이다")
CONFIG["C0"] = dict(max_dscore=d0)
CONFIG["C1"] = dict(max_dmetric=worst, where=worst_where, tol=TOL_IDENT)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【Q3-B】 ★★ C2 — 오라클 사전확률 팔. **여기를 C3 보다 먼저 읽는다**
run.log("\n" + "=" * 100)
run.log("【Q3-B】 C2 — **오라클 사전확률 팔**(전제의 직접 검정) · ★ 먼저 읽는다")
run.log("=" * 100)
run.log("  ▸ 진짜 유병률로 보정해도 전역 PR-AUC 가 안 오르면, 눈금 붕괴는")
run.log("    **사전확률 이동이 아니다** → 방법 A 는 그 자리에서 답이 나고 C3 의 비를 안 읽는다")

pooled_ap = lambda L: float(average_precision_score(Y_te, L))

# ── ★ 레코드 군집 부트스트랩 — 재표집 단위는 **레코드**다(비트가 아니다 · R11)
def cluster_boot(arms, seed, nb):
    """팔들을 **같은 재표집**에 태운다(짝지은 차를 위해). 반환: {arm: [ap...]}"""
    rng = np.random.RandomState(seed)
    recs = [int(r) for r in TE_R]
    out = {k: [] for k in arms}
    for _ in range(nb):
        pick = [recs[i] for i in rng.randint(0, len(recs), len(recs))]
        pos = np.concatenate([TE_POS[r] for r in pick])
        yy = Y_te[pos]
        if yy.sum() < 5 or yy.sum() == len(yy):
            continue
        for k, LL in arms.items():
            out[k].append(float(average_precision_score(yy, LL[pos])))
    return {k: np.asarray(v, float) for k, v in out.items()}

AP_RAW, AP_ORACLE = pooled_ap(L_raw), pooled_ap(L_oracle)
MACRO_RAW = float(np.mean(list(AP_raw_rec.values())))
run.log(f"\n  전역 PR-AUC — raw **{AP_RAW:.4f}** · oracle **{AP_ORACLE:.4f}** "
        f"(참고: 매크로 raw {MACRO_RAW:.4f} · 리듬 기저 교차환자 AUPRC {REF['base_auprc']})")

T0 = time.time()
BOOT = cluster_boot({"raw": L_raw, "oracle": L_oracle}, SEED0 + 11, NB_BOOT)
d_or = BOOT["oracle"] - BOOT["raw"]
GAIN_OR = AP_ORACLE - AP_RAW
or_lo, or_hi = float(np.percentile(d_or, 2.5)), float(np.percentile(d_or, 97.5))
OR_MDE = mde(or_lo, or_hi)
run.log(f"  ({time.time()-T0:.0f}초) 레코드 군집 부트스트랩 {len(d_or)}회 (레코드 {len(TE_R)}개 재표집)")
run.log(f"  C2 — 오라클 이득 **{GAIN_OR:+.4f}** [{or_lo:+.4f}, {or_hi:+.4f}] · MDE {OR_MDE:.4f}")

# ★ 통과 = 이득이 **자기 MDE 를 넘고** CI 가 0 을 뗄 것 (둘 다)
c2_ci = decide(or_lo, or_hi, 0.0, ">")
c2_mde = GAIN_OR > OR_MDE
C2_OK = c2_ci.startswith("✅") and c2_mde
run.log(f"    ▸ CI 판정 {c2_ci} · 이득 > MDE ? {'예' if c2_mde else '**아니오**'} "
        f"({GAIN_OR:+.4f} vs {OR_MDE:.4f})")
g_("C2", "✅ 지지" if C2_OK else ("❌ 기각" if c2_ci.startswith("❌") or not c2_mde else "⚠️ 미결"),
   "진짜 유병률 보정이 전역 PR-AUC 를 올린다 — **label shift 전제가 산다**" if C2_OK else
   "★★ 진짜 유병률로 보정해도 이득이 자기 MDE 를 못 넘는다 — "
   "**눈금 붕괴는 사전확률 이동이 아니다**")
run.log("  ⚠️ **오라클은 방법이 아니라 상한이다** — TEST 라벨의 유병률을 쓰므로 "
        "배포 가능한 절차가 아니고, **상한으로만** 인용한다")
if not C2_OK:
    run.log("\n  ⛔ **C2 ❌ → 사전등록대로 C3 의 회복률(비)을 읽지 않는다.**")
    run.log("     절대 ΔPR-AUC 는 계속 찍되(R41 ②), 회복률은 **미판독**으로 남긴다")
CONFIG["C2"] = dict(ap_raw=AP_RAW, ap_oracle=AP_ORACLE, gain=GAIN_OR,
                    lo=or_lo, hi=or_hi, mde=float(OR_MDE), ci_verdict=c2_ci,
                    gain_gt_mde=bool(c2_mde), passed=bool(C2_OK),
                    macro_raw=MACRO_RAW, n_boot=len(d_or))
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【Q3-C】 ★★★ C3 — EM 팔(주 관문) · C4 — 측정된 영점
run.log("\n" + "=" * 100)
run.log("【Q3-C】 C3 — **EM 팔**(주 관문) · C4 — **측정된** 영점")
run.log("=" * 100)

# ── EM (Saerens 2002). 확률 공간에서 π 만 추정한다(점수 변환은 로짓 시프트로 한다)
def em_prior(p, pi_tr, iters, tol, clip):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

# ── ★★ EM 하이퍼는 **DEV 에서 고정**한다 — 보고할 지표(TEST PR-AUC)로 고르면 누출(R22 · R36 ②)
DV_POS = {int(r): np.where(RID[DV_I] == r)[0] for r in DV_R}
PI_STAR_DEV = {int(r): float(TT[DV_I][DV_POS[int(r)]].mean()) for r in DV_R}
# ★ 클리핑 범위도 사전등록된 EM 하이퍼다. 약한 기저에서는 EM 이 **경계로 달아나므로**
#   (스모크 실측: π̂ 이 clip 하한 0.001 과 0.71 로 갈렸다) 격자가 그 축을 **감싸야** 한다(R33 ②).
GRID = [(it, tl, cp) for it in (100, 500) for tl in (1e-6, 1e-9)
        for cp in (1e-4, 1e-3, 1e-2, 5e-2)]
best, BEST_HP = None, None
for hp in GRID:
    err = np.mean([abs(em_prior(P_dev[DV_POS[int(r)]], PI_TR, *hp) - PI_STAR_DEV[int(r)])
                   for r in DV_R])
    if best is None or err < best:
        best, BEST_HP = err, hp
EM_ITERS, EM_TOL, PI_CLIP = BEST_HP
run.log(f"  EM 하이퍼(★ **DEV 에서 고정** · TEST 를 안 봤다) — iters={EM_ITERS} · "
        f"tol={EM_TOL:g} · clip={PI_CLIP:g} · DEV 평균 |π̂−π*| **{best:.5f}**")
# ★ DEV 에서조차 π̂ 이 안 맞으면 C3 의 실패는 **방법이 아니라 추정** 탓일 수 있다 —
#   나중에 C5 가 가르지만, 여기서 **미리 경고**해 둔다(요약이 사실과 어긋나지 않게).
DEV_SPREAD = float(np.std(list(PI_STAR_DEV.values())))
EM_SHAKY = best > DEV_SPREAD
if EM_SHAKY:
    run.log(f"    ⚠️ **DEV π̂ 오차({best:.5f})가 DEV 유병률 산포({DEV_SPREAD:.5f})보다 크다** — "
            "EM 추정이 이 기저 위에서 서지 않는다. C3 이 실패하면 **추정이 병목**일 수 있다(C5 에서 가른다)")

PI_HAT = {int(r): em_prior(P_te[TE_POS[int(r)]], PI_TR, EM_ITERS, EM_TOL, PI_CLIP)
          for r in TE_R}
L_em = apply_arm(PI_HAT)
AP_EM = pooled_ap(L_em)

B2 = cluster_boot({"raw": L_raw, "oracle": L_oracle, "em": L_em}, SEED0 + 11, NB_BOOT)
d_em = B2["em"] - B2["raw"]
GAIN_EM = AP_EM - AP_RAW
em_lo, em_hi = float(np.percentile(d_em, 2.5)), float(np.percentile(d_em, 97.5))
EM_MDE = mde(em_lo, em_hi)
run.log(f"\n  전역 PR-AUC — raw {AP_RAW:.4f} → **EM {AP_EM:.4f}** (oracle {AP_ORACLE:.4f})")
run.log(f"  ★ **절대 ΔPR-AUC(EM − raw) {GAIN_EM:+.4f}** [{em_lo:+.4f}, {em_hi:+.4f}] · "
        f"MDE {EM_MDE:.4f}")
run.log("    ▸ 절대 Δ 를 **항상 병기**한다 — 비만 보면 분모가 0 근처일 때 CI 가 폭발한다(R40 ② · R41 ②)")
c3_abs = decide(em_lo, em_hi, 0.0, ">")

# ── ★★★ 회복률 — **C2 가 통과했을 때만 읽는다**(사전등록된 판정 순서)
d_rec = (B2["em"] - B2["raw"]) / np.where(np.abs(B2["oracle"] - B2["raw"]) < 1e-9,
                                          np.nan, B2["oracle"] - B2["raw"])
d_rec = d_rec[np.isfinite(d_rec)]
REC_PT = GAIN_EM / GAIN_OR if abs(GAIN_OR) > 1e-9 else float("nan")
rc_lo = float(np.percentile(d_rec, 2.5)) if len(d_rec) > 20 else float("nan")
rc_hi = float(np.percentile(d_rec, 97.5)) if len(d_rec) > 20 else float("nan")
RATIO_READ = bool(VERD.get("C2", "").startswith("✅"))
if RATIO_READ:
    run.log(f"  회복률 = (EM−raw)/(oracle−raw) = **{REC_PT:.3f}** [{rc_lo:.3f}, {rc_hi:.3f}] "
            f"· 합격선 {RECOVERY_THR:.0%}  (분모 = **오라클 이득** · 사전 고정)")
    c3_ratio = np.isfinite(REC_PT) and REC_PT >= RECOVERY_THR
else:
    run.log(f"  ⛔ **회복률 미판독** — C2 가 통과하지 못했으므로 사전등록대로 비를 읽지 않는다.")
    run.log(f"     (참고로만: 점추정 {REC_PT:.3f} — 분모가 0 근처라 **수가 아니다**. 인용 금지)")
    c3_ratio = False

C3_OK = c3_abs.startswith("✅") and c3_ratio and RATIO_READ
if not RATIO_READ:
    g_("C3", "❌ 기각", "★★ C2 가 답을 냈다 — 오라클조차 못 올리므로 EM 이 올릴 것도 없다. "
                        "**방법 A 종결**")
elif C3_OK:
    g_("C3", "✅ 지지", f"절대 Δ 의 CI 가 0 을 뗐고 회복률 {REC_PT:.1%} ≥ {RECOVERY_THR:.0%} — **처방 확보**")
else:
    which = "절대 Δ" if not c3_abs.startswith("✅") else "회복률"
    g_("C3", "⚠️ 미결", f"둘 중 **{which}** 만 못 넘었다 — 하나만 만족하면 미결로 쓴다(등가가 아니다)")

# ── C4 — ★ 영점을 **측정**한다. 0 을 가정하지 않는다(R26 · R38 ②)
run.log(f"\n  C4 — 영점: 학습 라벨을 치환해 정보 없는 기저를 만들고 **같은 파이프라인**을 밟는다 "
        f"(reps={N_PERM})")
run.log("    ⚠️ 오라클 팔은 **진짜 유병률을 주입**하므로 영점에서도 이득이 날 수 있다 —")
run.log("       그게 이 관문이 필요한 이유다. 관측 이득은 **자기 영점 대비**로 읽는다")
T1 = time.time()
NUL = {"oracle": [], "em": [], "recovery": []}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    y_perm = TT[TR_I].astype(int)[rr.permutation(len(TR_I))]
    sc_ = fit_base(TR_I, y_perm)
    pd_, pt_ = sc_(DV_I), sc_(TE_I)
    cal_ = {"platt": make_platt, "isotonic": make_iso}[CAL_NAME](pd_, TT[DV_I])
    q_dev = np.clip(cal_(pd_), EPS, 1 - EPS); q_te = np.clip(cal_(pt_), EPS, 1 - EPS)
    pi_tr_n = float(TT[DV_I].mean())
    l0_ = logit(q_te)
    def arm_(pi_by):
        o = l0_.copy()
        for r in TE_R:
            p_ = TE_POS[int(r)]
            o[p_] = o[p_] + shift_logit(pi_by[int(r)], pi_tr_n)
        return o
    a_raw = pooled_ap(l0_)
    a_or = pooled_ap(arm_(PI_STAR))
    pih = {int(r): em_prior(q_te[TE_POS[int(r)]], pi_tr_n, EM_ITERS, EM_TOL, PI_CLIP)
           for r in TE_R}
    a_em = pooled_ap(arm_(pih))
    NUL["oracle"].append(a_or - a_raw); NUL["em"].append(a_em - a_raw)
    NUL["recovery"].append((a_em - a_raw) / (a_or - a_raw) if abs(a_or - a_raw) > 1e-9 else np.nan)
nm_or, nlo_or, nhi_or, _ = boot_mean(NUL["oracle"], SEED0 + 61)
nm_em, nlo_em, nhi_em, _ = boot_mean(NUL["em"], SEED0 + 62)
run.log(f"  ({time.time()-T1:.0f}초) 영점 이득 — oracle {nm_or:+.4f} [{nlo_or:+.4f}, {nhi_or:+.4f}] · "
        f"em {nm_em:+.4f} [{nlo_em:+.4f}, {nhi_em:+.4f}]")
run.log(f"    관측 — oracle {GAIN_OR:+.4f} · em {GAIN_EM:+.4f}")
or_over = np.isfinite(nhi_or) and GAIN_OR > nhi_or
em_over = np.isfinite(nhi_em) and GAIN_EM > nhi_em
run.log(f"    영점 초과? oracle {'✅' if or_over else '⚠️ 아니다'} · em {'✅' if em_over else '⚠️ 아니다'}")
g_("C4", "✅ 지지" if (or_over or em_over) else "⚠️ 미결",
   "관측 이득이 **측정된 영점을 넘는다**" if (or_over or em_over) else
   "★★ 관측 이득이 영점 범위 안이다 — 이득이 **모형이 아니라 유병률 주입**에서 온다. "
   "그러면 위 팔들을 눈금 수리의 증거로 읽지 않는다")
CONFIG["C3"] = dict(ap_em=AP_EM, gain=GAIN_EM, lo=em_lo, hi=em_hi, mde=float(EM_MDE),
                    abs_verdict=c3_abs, recovery=REC_PT, rec_lo=rc_lo, rec_hi=rc_hi,
                    ratio_read=RATIO_READ, ratio_pass=bool(c3_ratio), passed=bool(C3_OK),
                    em_iters=EM_ITERS, em_tol=EM_TOL, pi_clip=PI_CLIP, dev_pi_err=float(best))
CONFIG["C4"] = dict(n_perm=N_PERM, oracle=dict(mean=nm_or, lo=nlo_or, hi=nhi_or),
                    em=dict(mean=nm_em, lo=nlo_em, hi=nhi_em),
                    oracle_over=bool(or_over), em_over=bool(em_over))
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【Q3-D】 C5 π̂ 진단 · C6 매크로(항등 검산) · 필요표본 · ★ C7 결론 검산표
run.log("\n" + "=" * 100)
run.log("【Q3-D】 C5 — π̂ 진단 · C6 — 매크로 항등 · C7 — 결론 검산표")
run.log("=" * 100)

# ── C5 — 관문이 아니다. **C2 ✅ · C3 ❌ 일 때 「방법 vs 추정」을 가른다**
err = np.array([abs(PI_HAT[int(r)] - PI_STAR[int(r)]) for r in TE_R])
rel = np.array([PI_HAT[int(r)] / max(PI_STAR[int(r)], 1e-9) for r in TE_R])
rho_pi = spearman([PI_STAR[int(r)] for r in TE_R], [PI_HAT[int(r)] for r in TE_R])
em_, elo_, ehi_, _ = boot_mean(err, SEED0 + 71)
run.log(f"  C5 — |π̂ − π*| 평균 **{em_:.5f}** [{elo_:.5f}, {ehi_:.5f}] · 중앙 {np.median(err):.5f} · "
        f"최대 {err.max():.5f}")
run.log(f"       ρ(π̂, π*) = **{rho_pi:+.4f}** · π̂/π* 중앙 {np.median(rel):.3f} · "
        f"π* 범위 {min(PI_STAR.values()):.4f}~{max(PI_STAR.values()):.4f}")
run.log(f"       기저 보정도 ECE — DEV {ECE_DEV:.5f} · TEST {ECE_TE:.5f} "
        f"(보정기 = {CAL_NAME} · **DEV 에서만** 적합)")
PI_BLAME = (rho_pi < 0.5) or (em_ > 0.5 * float(np.mean(list(PI_STAR.values()))))
if VERD.get("C2", "").startswith("✅") and not VERD.get("C3", "").startswith("✅"):
    run.log(f"  ★ C2 ✅ · C3 ❌/⚠️ 상황 — C5 의 판독: "
            + ("**추정이 병목**(π̂ 오차가 크다) → 추정기 교체(BBSE·MLLS)가 정당해진다"
               if PI_BLAME else
               "**π̂ 는 무죄** → 모형 오설정(label shift 가정 실패)으로 읽고 한계를 명시한다"))
else:
    run.log("  ▸ (C5 의 갈림 판독은 **C2 ✅ · C3 ❌** 일 때만 쓴다 — 지금은 해당 없음)")

# ── C6 — 매크로. ★ 판정 지표가 아니라 **항등 검산**이다
run.log(f"\n  C6 — 매크로(레코드 평균). ★ 판정 지표가 아니라 **항등 대조**다")
MACRO = {}
for nm, LL in (("raw", L_raw), ("identity", L_ident), ("oracle", L_oracle), ("em", L_em)):
    ap_, au_ = per_record(LL)
    MACRO[nm] = dict(prauc=float(np.mean(list(ap_.values()))),
                     auroc=float(np.mean(list(au_.values()))), n=len(ap_))
    flag = "단조" if ARM_MONOTONE[nm] else "**비단조 — 진짜 판정 대상**"
    run.log(f"    {nm:<9}매크로 PR-AUC {MACRO[nm]['prauc']:.6f} · AUROC {MACRO[nm]['auroc']:.6f}  ({flag})")
mx = max(abs(MACRO[k]["prauc"] - MACRO["raw"]["prauc"]) for k in MACRO)
mx = max(mx, max(abs(MACRO[k]["auroc"] - MACRO["raw"]["auroc"]) for k in MACRO))
run.log(f"    max|Δ매크로| = **{mx:.1e}** (허용 {TOL_IDENT:.0e})")
g_("C6", "✅ 지지" if mx < TOL_IDENT else "❌ 기각",
   "전 팔이 단조라 매크로가 **정확히 불변**이다 — 사전등록의 「매크로 비열등」은 항등으로 통과"
   if mx < TOL_IDENT else "매크로가 움직였다 — **버그이거나 비단조**다")
if mx >= TOL_IDENT:
    raise AssetError("C6 실패 — 단조 팔인데 매크로가 움직였다. C1 과 모순이므로 중단")

# ── 필요표본 — ★ 프레임을 밝히고, 효과가 0 근처면 **해석 불가**라고 같이 찍는다
run.log(f"\n  필요표본 (**우월 프레임** · 단위 = **TEST 레코드** · 현재 {len(TE_R)}개)")
run.log(f"  {'대상':<12}{'효과':>10}{'반폭':>9}{'우월50%':>10}{'우월80%':>10}  비고")
NEED = {}
for nm, eff, half_ in (("C2 oracle", GAIN_OR, OR_MDE), ("C3 EM", GAIN_EM, EM_MDE)):
    n5 = need_super(len(TE_R), half_, eff, False); n8 = need_super(len(TE_R), half_, eff, True)
    zero = abs(eff) < half_          # ★ 효과가 자기 반폭 안 = 0 근처
    NEED[nm] = dict(effect=float(eff), half=float(half_), sup50=float(n5), sup80=float(n8),
                    uninterpretable=bool(zero))
    run.log(f"  {nm:<12}{eff:>+10.4f}{half_:>9.4f}{n5:>10.0f}{n8:>10.0f}  "
            + ("★ **효과 ≈ 0 이라 해석 불가**(R41 ②)" if zero else "읽을 수 있다"))
run.log("    ▸ 판정은 필요표본이 아니라 **MDE 로** 한다 — 「좁은 CI 안의 0」과 "
        "「넓은 CI 안의 0」은 다른 말이다(R41 ②)")

# ── ★ C7 결론 검산표 (R38 ⑦ · R39 ⑤)
run.log("\n  ★ C7 — **결론 검산표**")
CHECK = [
    dict(claim=f"C0·C1 항등이 성립한다 (max|Δscore| {d0:.0e} · max|Δ매크로| {mx:.0e})",
         num="보정을 **로짓 공간의 상수 시프트**로 구현했고 π=π_tr 에서 시프트가 정확히 0.0",
         assume="**없음** — 구성으로 보장되고 런타임에 검사한다(R34 ③ · R35 ④)",
         iffalse="—"),
    dict(claim=f"C2 오라클 이득 {GAIN_OR:+.4f} [{or_lo:+.4f}, {or_hi:+.4f}]",
         num=f"MDE {OR_MDE:.4f} · 이득>MDE {'예' if c2_mde else '아니오'} · 영점 상단 {nhi_or:+.4f}",
         assume="오라클 유병률이 **레코드 안에서 상수**라는 것(홀터 전체를 한 분포로 본다)",
         iffalse="유병률이 기록 안에서 표류하면 오라클조차 상한이 아니다 — 상한이 **더 높다**"),
    dict(claim=(f"C3 EM 절대 Δ {GAIN_EM:+.4f} [{em_lo:+.4f}, {em_hi:+.4f}]"
                + (f" · 회복률 {REC_PT:.3f}" if RATIO_READ else " · 회복률 **미판독**")),
         num=(f"분모 = 오라클 이득 {GAIN_OR:+.4f} (사전 고정)" if RATIO_READ else
              f"C2 가 통과하지 못해 **비를 읽지 않는다** — 분모 {GAIN_OR:+.4f} 가 0 근처다"),
         assume="label shift(`p(x|y)` 불변). SVDB 는 **covariate shift 도 있어** 어긋날 수 있다",
         iffalse="가정이 깨지면 EM 의 π̂ 가 편향된다 → C5 가 「방법 vs 추정」을 가른다"),
    dict(claim=f"C4 영점 — oracle {nm_or:+.4f} · em {nm_em:+.4f}",
         num=f"학습 라벨 치환 reps={N_PERM} · **같은 파이프라인**(DEV 보정 → 레코드별 EM)",
         assume="**없음** — 라벨만 바꿔 같은 절차를 밟는다",
         iffalse="—  ⚠️ 단, 오라클 팔은 진짜 유병률을 주입하므로 영점에서도 이득이 날 수 있다"),
    dict(claim="매크로는 **판정 지표가 아니다**",
         num=f"전 팔 단조 → C1·C6 에서 max|Δ| < {TOL_IDENT:.0e} (항등)",
         assume="**없음** — 상수 로짓 시프트의 성질이다",
         iffalse="비단조 팔(클리핑·레코드별 임계값)을 넣는 후속 런에서만 매크로가 판정 대상"),
    dict(claim=f"전역이 주 지표인 **예외 런**이다 (지배 지분 {DOMINANT:.3f})",
         num=f"TEST 레코드 {len(TE_R)} · 제외 {len(EXCL)} · GMIN_S = S≥{MIN_S} & N≥{MIN_N}",
         assume="Q3 의 표적이 **환자 간 점수 비교 가능성**이라 pooled 에서만 보인다는 것",
         iffalse="그래도 **전역 단독 인용은 금지**다 — 지배 지분·제외와 함께만 쓴다(R11)"),
    dict(claim="보정기·EM 하이퍼 선택에 누출이 없다",
         num=f"보정기 {CAL_NAME} 는 **DEV 반쪽 홀드아웃** ECE {ECE_DEV:.5f} 로, "
             f"EM 하이퍼는 **DEV** |π̂−π*| {CONFIG['C3']['dev_pi_err']:.5f} 로 골랐다",
         assume="**없음** — TEST 지표를 선택에 쓰지 않았다(R22 · R36 ②)",
         iffalse=f"⚠️ DEV **자기적합** ECE 는 {ECE_DEV_IN:.5f} 로 구성상 0 에 가깝다 — 그 수는 증거가 아니다"),
    dict(claim=("★★ C2 의 이득이 **자기 영점을 넘는가** — 넘지 못하면 눈금 수리의 증거가 아니다"
                if not or_over else "C2 의 이득이 자기 영점을 넘는다"),
         num=f"관측 {GAIN_OR:+.4f} vs 영점 상단 {nhi_or:+.4f} → {'초과' if or_over else '**영점 안**'}",
         assume="영점이 관측과 **같은 파이프라인**을 밟았다는 것(라벨만 치환)",
         iffalse=("★★ 오라클 팔은 **진짜 유병률을 주입**하므로 정보 없는 기저에서도 이득이 난다. "
                  "이득이 영점 안이면 그건 **모형의 눈금을 고친 것이 아니라 유병률을 주입한 것**이고, "
                  "C2 ✅ 를 처방의 증거로 읽으면 안 된다"
                  if not or_over else "—")),
]
for i, c in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{c['claim']}**")
    run.log(f"      근거   {c['num']}")
    run.log(f"      가정   {c['assume']}")
    run.log(f"      틀리면 {c['iffalse']}")
CONFIG["C5"] = dict(pi_err_mean=em_, lo=elo_, hi=ehi_, pi_err_max=float(err.max()),
                    rho=float(rho_pi), ece_dev=ECE_DEV, ece_test=ECE_TE,
                    calibrator=CAL_NAME, pi_blame=bool(PI_BLAME))
CONFIG["C6"] = dict(macro=MACRO, max_delta=float(mx))
CONFIG["need"] = NEED; CONFIG["C7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【Q3-E】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례·제목은 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① 기저 보정도 — 신뢰도 곡선(C5). EM 은 보정된 사후확률을 전제한다
for nm, pp, yy, c in (("DEV", P_dev, TT[DV_I], "tab:blue"),
                      ("TEST", P_te, Y_te, "tab:red")):
    q = np.unique(np.quantile(pp, np.linspace(0, 1, 11)))
    if len(q) > 2:
        b = np.clip(np.digitize(pp, q[1:-1]), 0, len(q) - 2)
        xs = [pp[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]
        ys = [np.asarray(yy, float)[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]
        ax[0].plot(xs, ys, "o-", color=c, ms=4, label=f"{nm} (ECE {ece_of(pp, np.asarray(yy, float)):.4f})")
lim = max(P_dev.max(), P_te.max(), 0.05)
ax[0].plot([0, lim], [0, lim], "k--", lw=.9, label="ideal")
ax[0].set_xlabel("predicted probability"); ax[0].set_ylabel("observed frequency")
ax[0].set_title(f"C5 : base calibration ({CAL_NAME}, fit on DEV only)", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② 주 지표 — 전역 PR-AUC 이득(관측 vs 영점)
labs = ["oracle\n(upper bound)", "EM"]
obs = [GAIN_OR, GAIN_EM]
lo_ = [GAIN_OR - or_lo, GAIN_EM - em_lo]; hi_ = [or_hi - GAIN_OR, em_hi - GAIN_EM]
ax[1].errorbar(np.arange(2) - 0.08, obs, yerr=[lo_, hi_], fmt="o", capsize=5,
               color="tab:red", label="observed")
ax[1].errorbar(np.arange(2) + 0.08, [nm_or, nm_em],
               yerr=[[nm_or - nlo_or, nm_em - nlo_em], [nhi_or - nm_or, nhi_em - nm_em]],
               fmt="x", capsize=4, color="tab:gray", label="label-permutation null")
ax[1].axhline(0, color="k", lw=.9)
ax[1].set_xticks(range(2)); ax[1].set_xticklabels(labs, fontsize=8)
ax[1].set_ylabel("pooled dPR-AUC vs raw")
ax[1].set_title(f"C2 (read first) / C3 : raw={AP_RAW:.4f}", fontsize=9)
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, axis="y")

# ③ C5 — π̂ vs π*
xs = [PI_STAR[int(r)] for r in TE_R]; ys = [PI_HAT[int(r)] for r in TE_R]
ax[2].scatter(xs, ys, s=28, color="tab:purple")
m_ = max(max(xs), max(ys)) * 1.05
ax[2].plot([0, m_], [0, m_], "k--", lw=.9)
ax[2].axhline(PI_TR, ls=":", color="tab:blue", lw=1.0, label=f"pi_tr = {PI_TR:.4f}")
ax[2].set_xlabel("true record prevalence pi*"); ax[2].set_ylabel("EM estimate pi_hat")
ax[2].set_title(f"C5 : rho = {rho_pi:+.3f}, mean |err| = {em_:.4f}", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q3_prior_em", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:7]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)' if g == 'C5' else '(미실행)')}")
run.log("")
run.log(f"  전역 PR-AUC — raw {AP_RAW:.4f} · oracle {AP_ORACLE:.4f} · EM {AP_EM:.4f}")
run.log(f"  ★ 판정 순서대로 **C2 를 먼저** 읽었다 — {VERD.get('C2', '?')}")
# ★★ C2 가 ✅ 여도 **자기 영점을 못 넘었으면** 처방의 증거가 아니다. 요약이 그걸 삼키면 안 된다
if ok_("C2") and not or_over:
    run.log("")
    run.log(f"  ⚠️⚠️ **경고 — C2 의 이득 {GAIN_OR:+.4f} 이 자기 영점(상단 {nhi_or:+.4f}) 안이다.**")
    run.log("     오라클 팔은 **진짜 유병률을 주입**하므로 정보 없는 기저에서도 이득이 난다.")
    run.log("     즉 이 이득은 **모형의 눈금을 고친 증거가 아니다**. C2 ✅ 를 그렇게 읽지 마라")
run.log("")
if not ok_("C2"):
    run.log("  ⛔⛔ **C2 ❌ — 방법 A 를 종결한다.**")
    run.log("     최종 문장:")
    run.log("       「SVDB 2리드 홀터 SVEB 검출에서, 기록별 사전확률 보정은 **진짜 유병률을")
    run.log(f"        써도**(오라클 상한) 전역 PR-AUC 를 {GAIN_OR:+.4f} "
            f"[{or_lo:+.4f}, {or_hi:+.4f}] 밖에 움직이지 못한다.")
    run.log("        따라서 눈금 붕괴는 **사전확률 이동으로 설명되지 않는다**.")
    run.log("        레코드 내 순위는 정의상 보존되므로(C0·C1 항등) 매크로는 불변이다.」")
    run.log("     → **방법 B(Q4 `burden-feature`)로 간다.** Q7-AA 가 닫은 건 표적 모집단으로서의")
    run.log("        burden 이지 **특징으로서의 burden 이 아니다**(AA4 의 ρ +0.2898 이 영점 위)")
elif ok_("C3"):
    run.log("  ★★★ **처방을 확보했다.** 무라벨 EM 사전확률 보정이 전역 PR-AUC 를 회복시킨다")
    run.log(f"     회복률 {REC_PT:.1%} (분모 = 오라클 이득 · 사전 고정) · "
            f"절대 Δ {GAIN_EM:+.4f} [{em_lo:+.4f}, {em_hi:+.4f}]")
    run.log("     → **Q4 는 이제 「A 대비 증분」으로 판정한다**(A 를 기저로 깐다)")
    if not em_over:
        run.log("     ⚠️⚠️ **단, EM 이득이 자기 영점을 못 넘었다**(C4) — 처방으로 쓰기 전에")
        run.log("        영점부터 규명해야 한다. 이득이 모형이 아니라 절차에서 왔을 수 있다")
else:
    run.log("  ⚠️ **C2 ✅ · C3 미결/❌ — 전제는 사는데 처방이 안 선다.**")
    run.log(f"     C5 판독: " + ("**추정이 병목** → 사전확률 추정기 교체(BBSE·MLLS)가 정당해진다"
                                 if PI_BLAME else
                                 "**π̂ 무죄 → 모형 오설정**(label shift 가정 실패). 한계 명시 후 Q4"))
    run.log(f"     ⚠️ 미결은 **등가가 아니다** — 있어도 ΔPR-AUC CI 상단 {em_hi:+.4f} 이하다(R36 ①)")
run.log("")
run.log("  ▸ ★ **전역 단독 인용 금지** — 지배 지분 "
        f"{DOMINANT:.3f} · 제외 {len(EXCL)} · GMIN_S(S≥{MIN_S}, N≥{MIN_N}) 와 함께만(R11)")
run.log("  ▸ ★ **오라클은 방법이 아니라 상한**이다 — TEST 유병률을 쓰므로 배포 불가")
run.log("  ▸ ★ 매크로는 **항등 대조**였다(판정 지표가 아니다) — 움직였으면 버그였다")

run.finish({
    "exp_id": "quest46_q3_prior_em",
    "metric": "pooled_prauc_delta_em",
    "value": float(GAIN_EM),
    "passed": bool(ok_("C0") and ok_("C1") and ok_("C2") and ok_("C3")),
    "summary": ("층② 눈금의 첫 처방 — 무라벨 EM 사전확률 보정(Saerens 2002). "
                "C2(오라클 사전확률 팔)를 C3 보다 **먼저** 읽어 label shift 전제를 직접 검정하고, "
                "기저 보정은 **DEV 에서만** 적합했으며, 매크로는 판정이 아니라 **항등 대조**다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "C0": CONFIG.get("C0", {}), "C1": CONFIG.get("C1", {}),
    "C2": CONFIG.get("C2", {}), "C3": CONFIG.get("C3", {}), "C4": CONFIG.get("C4", {}),
    "C5": CONFIG.get("C5", {}), "C6": CONFIG.get("C6", {}), "C7": CONFIG.get("C7", []),
    "need": CONFIG.get("need", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q3_prior_em.ipynb`")
